# EDA và trực quan hóa Penn Action & J-HMDB

Notebook khảo sát annotation gốc ở mức **video/chuỗi**, phân phối hành động, số frame, split, chất lượng dữ liệu và một số ví dụ skeleton. Không chạy mô hình hoặc sửa dữ liệu. Chạy các ô theo thứ tự với kernel Python của `.venv`. Nếu mở từ thư mục dự án: ` .\.venv\Scripts\python.exe -m jupyter lab`.

Penn dùng split chính thức trong trường `train`; J-HMDB dùng split chính thức số 1. Số frame dưới đây được lấy từ annotation, không phải thời lượng giây. Các tỷ lệ visibility chỉ có ý nghĩa với Penn; không suy diễn visibility của J-HMDB từ tọa độ.


In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = next((p for p in ROOT.parents if (p / "data" / "raw").exists()), ROOT)
PENN = ROOT / "data" / "raw" / "Penn_Action"
JHMDB = ROOT / "data" / "raw" / "jhmdb"
print("Python:", sys.executable, "\nThư mục dự án:", ROOT)
for path in [PENN / "labels", JHMDB / "joint_positions", JHMDB / "splits"]:
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {path}; hãy mở notebook trong dự án hoặc sửa ROOT."
        )
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.2})

## 1. Đọc metadata và kiểm tra dữ liệu

Mỗi dòng tương ứng một annotation; các lỗi đọc được giữ trong bảng riêng. Với J-HMDB, split 1 được ghép bằng tên lớp và tên video, nhờ đó phân biệt các video trùng tên thuộc lớp khác nhau.

In [ ]:
def text_mat(value):
    return str(np.asarray(value).squeeze().item())


def scalar(value):
    return int(np.asarray(value).squeeze())


penn_rows, j_rows, errors = [], [], []
for path in sorted((PENN / "labels").glob("*.mat")):
    try:
        a = sio.loadmat(path)
        x, y, vis = a["x"], a["y"], a["visibility"]
        if x.ndim != 2 or x.shape != y.shape or x.shape != vis.shape:
            raise ValueError("Kích thước x/y/visibility không khớp")
        if x.shape[1] != 13 and x.shape[0] == 13:
            x, y, vis = x.T, y.T, vis.T
        if x.shape[1] != 13:
            raise ValueError(f"Số khớp không phải 13: {x.shape}")
        nframes = scalar(a["nframes"])
        train = scalar(a["train"])
        penn_rows.append(
            dict(
                dataset="Penn Action",
                action=text_mat(a["action"]),
                video_id=path.stem,
                split={1: "train", -1: "test", 0: "test"}.get(train, f"khác ({train})"),
                frames=nframes,
                annotated_frames=x.shape[0],
                visible_ratio=np.mean(vis == 1),
                nonfinite_coords=int((~np.isfinite(x) | ~np.isfinite(y)).sum()),
                frame_data_exists=(PENN / "frames" / path.stem).is_dir(),
                mat_path=path,
            )
        )
    except Exception as exc:
        errors.append(dict(dataset="Penn Action", path=str(path), error=str(exc)))

split_map = {}
for path in sorted((JHMDB / "splits").glob("*_test_split1.txt")):
    action = path.name.removesuffix("_test_split1.txt")
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) == 2:
            split_map[(action, Path(parts[0]).stem)] = {"1": "train", "2": "test"}.get(
                parts[1], "khác"
            )
for path in sorted((JHMDB / "joint_positions").glob("*/*/joint_positions.mat")):
    action, video_id = path.parent.parent.name, path.parent.name
    if action.startswith(".") or video_id.startswith("."):
        continue
    try:
        pos = sio.loadmat(path)["pos_img"]
        if pos.ndim != 3 or pos.shape[:2] != (2, 15):
            raise ValueError(f"Kích thước pos_img không phải (2, 15, T): {pos.shape}")
        j_rows.append(
            dict(
                dataset="J-HMDB",
                action=action,
                video_id=video_id,
                split=split_map.get((action, video_id), "không có split"),
                frames=pos.shape[2],
                nonfinite_coords=int((~np.isfinite(pos).all(axis=0)).sum()),
                frame_data_exists=(
                    JHMDB / "ReCompress_Videos" / action / (video_id + ".avi")
                ).is_file(),
                mat_path=path,
            )
        )
    except Exception as exc:
        errors.append(dict(dataset="J-HMDB", path=str(path), error=str(exc)))

penn = pd.DataFrame(penn_rows)
jhmdb = pd.DataFrame(j_rows)
if penn.empty or jhmdb.empty:
    raise ValueError(
        f"Không đọc được annotation: Penn={len(penn)}, J-HMDB={len(jhmdb)}; lỗi: {errors[:5]}"
    )
all_data = pd.concat([penn, jhmdb], ignore_index=True)
print("Annotation đọc thành công:", len(all_data), "| Lỗi:", len(errors))
display(pd.DataFrame(errors, columns=["dataset", "path", "error"]).head(20))

In [ ]:
summary = all_data.groupby("dataset").agg(
    so_chuoi=("video_id", "size"),
    so_lop=("action", "nunique"),
    tong_frame=("frames", "sum"),
    frame_trung_vi=("frames", "median"),
    thieu_anh_video=("frame_data_exists", lambda s: (~s).sum()),
    toa_do_khong_huu_han=("nonfinite_coords", "sum"),
)
display(summary)
print("Số lượng theo split (annotation đã đọc):")
display(pd.crosstab(all_data.dataset, all_data.split))
print(
    "Penn: số chuỗi có nframes khác số hàng annotation:",
    (penn.frames != penn.annotated_frames).sum(),
)
print(
    "J-HMDB: số mục split 1 không tìm thấy annotation:",
    len(set(split_map) - set(zip(jhmdb.action, jhmdb.video_id))),
)

## 2. Phân phối lớp và split

So sánh số lượng video của từng lớp trong từng bộ. Tên hành động giữa hai bộ không nhất thiết đồng nhất.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, (name, group) in zip(axes, all_data.groupby("dataset", sort=False)):
    counts = pd.crosstab(group.action, group.split)
    counts = counts.loc[counts.sum(axis=1).sort_values().index]
    counts.plot.barh(stacked=True, ax=ax, width=0.8)
    ax.set(title=f"{name}: số chuỗi theo hành động và split", xlabel="Số chuỗi", ylabel="Hành động")
    ax.legend(title="Split")
plt.tight_layout()
plt.show()

## 3. Độ dài chuỗi và visibility

Visibility là tỷ lệ cờ `visibility == 1` trên toàn bộ khớp-frame trong từng annotation Penn; đây không phải confidence của mô hình.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, group in all_data.groupby("dataset"):
    axes[0].hist(group.frames, bins=35, alpha=0.55, label=name)
axes[0].set(title="Phân phối số frame/chuỗi", xlabel="Số frame theo annotation", ylabel="Số chuỗi")
axes[0].legend()
axes[1].hist(penn.visible_ratio, bins=25, color="teal")
axes[1].set(
    title="Penn: tỷ lệ khớp-frame nhìn thấy", xlabel="Tỷ lệ visibility = 1", ylabel="Số chuỗi"
)
plt.tight_layout()
plt.show()
display(all_data.groupby("dataset").frames.describe().round(2))

## 4. Minh họa annotation trên frame giữa

Ví dụ đầu tiên có thể đọc được trong mỗi bộ; đường nối và tên khớp tuân theo thứ tự annotation. Nếu ảnh hoặc video thiếu, phần tương ứng sẽ thông báo.

In [ ]:
penn_edges = [
    (0, 1),
    (0, 2),
    (1, 2),
    (1, 3),
    (3, 5),
    (2, 4),
    (4, 6),
    (1, 7),
    (2, 8),
    (7, 8),
    (7, 9),
    (9, 11),
    (8, 10),
    (10, 12),
]
j_edges = [
    (0, 1),
    (0, 2),
    (0, 3),
    (0, 4),
    (3, 7),
    (7, 11),
    (4, 8),
    (8, 12),
    (1, 5),
    (1, 6),
    (5, 9),
    (9, 13),
    (6, 10),
    (10, 14),
]


def overlay(ax, image, xy, valid, edges, title):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    valid = np.asarray(valid, dtype=bool) & np.isfinite(xy).all(axis=1)
    for a, b in edges:
        if valid[a] and valid[b]:
            ax.plot(xy[[a, b], 0], xy[[a, b], 1], color="lime", linewidth=2)
    ax.scatter(xy[valid, 0], xy[valid, 1], c="red", s=20)
    ax.set_title(title)
    ax.axis("off")


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, group) in zip(axes, [("Penn Action", penn), ("J-HMDB", jhmdb)]):
    shown = False
    for row in group[group.frame_data_exists].itertuples():
        a = sio.loadmat(row.mat_path)
        if name == "Penn Action":
            x, y, v = a["x"], a["y"], a["visibility"]
            if x.shape[1] != 13:
                x, y, v = x.T, y.T, v.T
            t = x.shape[0] // 2
            image = cv2.imread(str(PENN / "frames" / row.video_id / f"{t + 1:06d}.jpg"))
            xy = np.column_stack([x[t], y[t]])
            valid = v[t] == 1
            edges = penn_edges
        else:
            pos = a["pos_img"]
            t = pos.shape[2] // 2
            cap = cv2.VideoCapture(
                str(JHMDB / "ReCompress_Videos" / row.action / (row.video_id + ".avi"))
            )
            cap.set(cv2.CAP_PROP_POS_FRAMES, t)
            ok, image = cap.read()
            cap.release()
            if not ok:
                image = None
            xy = pos[:, :, t].T
            valid = np.ones(15, dtype=bool)
            edges = j_edges
        if image is not None:
            overlay(
                ax,
                image,
                xy,
                valid,
                edges,
                f"{name}: {row.action} | {row.video_id} | frame {t + 1}",
            )
            shown = True
            break
    if not shown:
        ax.text(0.5, 0.5, f"{name}: không đọc được frame", ha="center")
        ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Kiểm tra nhanh các trường hợp cần xem lại

Thiếu ảnh/video chỉ kiểm tra sự tồn tại thư mục hoặc tệp; không xác nhận đủ mọi frame hay tính toàn vẹn của video. Tọa độ không hữu hạn được đếm theo khớp-frame.

In [ ]:
issues = all_data[
    (~all_data.frame_data_exists)
    | (all_data.nonfinite_coords > 0)
    | (all_data.split == "không có split")
].copy()
print(f"Số annotation có ít nhất một dấu hiệu cần kiểm tra: {len(issues)}")
display(issues.drop(columns=["mat_path"]).head(30))